In [1]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

BASE_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SOURCE = BASE_DIR / "database_final.xlsx"
SHEET_NAME = 0
HEADER_ROW = 0
MODE = "final"  # "raw" or "final"
REPORT_DIR = BASE_DIR / "outputs" / "tables" / "technical_validation"
SUM_TOLERANCE = 0.5
SUM_FIELD_TOLERANCE = 1e-6
DUPLICATE_DECIMALS = 6

ELEMENTS = ["Ni", "Co", "Fe", "Cr", "Al", "Ti", "Cu", "V", "Mo", "Hf", "W", "Ta", "B", "C", "Nb", "Mn"]
STEPS = ["HT 1", "Recrystallization", "Aging 1", "Aging 2"]
PROCESS = [c for s in STEPS for c in (f"{s}, °C", f"{s}, min")] + ["Cold deformation, %"]
COOLING = ["Cooling_after_HT1", "Cooling_after_Recryst", "Cooling_after_Aging1"]
REQUIRED = ["Paper", "Composition, at.%", "Preparation", "Phase composition", "DOI"]
FLAGS = ["L12 presence", "Target state"]
OPTIONAL_NUM = ["Solvus Temperature, °C", "L12 fraction, %"]
EXPECTED = REQUIRED + ELEMENTS + ["Sum"] + PROCESS + COOLING + FLAGS + OPTIONAL_NUM
ALIASES = {"Composition (at.%)": "Composition, at.%"}

SAMPLE_ID_PATTERN = r"S[1-9]\d*"   # experimental samples of this work: S1, S2, ...
SELF_DOI = "this work"              # normalised DOI of the experimental records


In [2]:
def is_missing(value):
    if pd.isna(value):
        return True
    return isinstance(value, str) and value.strip().casefold() in {"", "nan", "none", "null"}


def normalise_phase(value):
    if is_missing(value):
        return None
    text = unicodedata.normalize("NFKC", str(value)).strip()
    parts = [p.strip() for p in text.split("+")]
    if not all(parts):
        return None
    return frozenset(parts)


def normalise_doi(value):
    if is_missing(value):
        return None
    text = str(value).strip().casefold()
    return re.sub(r"^(?:https?://(?:dx\.)?doi\.org/|doi\s*:\s*)", "", text).strip()


def validate_dataset(frame, mode="raw", excel_first_data_row=2):
    if mode not in {"raw", "final"}:
        raise ValueError("MODE must be 'raw' or 'final'")
    df = frame.copy(deep=True).reset_index(drop=True)
    df.columns = [str(c).strip() for c in df.columns]
    if df.columns.duplicated().any():
        raise ValueError(f"Duplicate headers after trimming: {df.columns[df.columns.duplicated()].tolist()}")
    renamed = {}
    for old, new in ALIASES.items():
        if old in df.columns and new not in df.columns:
            df = df.rename(columns={old: new})
            renamed[old] = new
    n = len(df)
    missing = {c: df[c].map(is_missing) for c in df.columns}
    numeric = {c: pd.to_numeric(df[c].mask(missing[c]), errors="coerce")
               for c in ELEMENTS + ["Sum"] + PROCESS + FLAGS + OPTIONAL_NUM if c in df.columns}
    summaries, details = [], []

    # Experimental samples of this work (Paper = S1, S2, ...) are not literature papers
    if "Paper" in df.columns:
        paper_text = df["Paper"].map(lambda v: "" if is_missing(v) else str(v).strip())
        sample_mask = paper_text.str.fullmatch(SAMPLE_ID_PATTERN).fillna(False).astype(bool)
    else:
        paper_text = pd.Series("", index=df.index)
        sample_mask = pd.Series(False, index=df.index)

    def add(name, field, bad=None, eligible=None, severity="ERROR", note="", observed=None, expected=None):
        if eligible is None:
            eligible = pd.Series(True, index=df.index)
        eligible = pd.Series(eligible, index=df.index).fillna(False).astype(bool)
        if bad is None:
            bad = pd.Series(False, index=df.index)
        bad = pd.Series(bad, index=df.index).fillna(False).astype(bool) & eligible
        count = int(bad.sum())
        status = (severity if count else ("PASS" if eligible.any() else "NOT_EVALUATED"))
        summaries.append({"Check": name, "Field": field, "Rows_evaluated": int(eligible.sum()),
                          "Issues": count, "Status": status, "Note": note})
        for i in df.index[bad]:
            row = {"Check": name, "Severity": severity, "Field": field,
                   "Record_position": int(i + 1), "Excel_row": int(excel_first_data_row + i),
                   "Observed": observed.loc[i] if isinstance(observed, pd.Series) else (observed if observed is not None else (df.at[i, field] if field in df.columns else "")),
                   "Expected": expected.loc[i] if isinstance(expected, pd.Series) else expected,
                   "Note": note}
            for c in REQUIRED:
                row[c] = df.at[i, c] if c in df.columns else None
            details.append(row)

    for c in EXPECTED:
        if c not in df.columns:
            summaries.append({"Check": "Schema", "Field": c, "Rows_evaluated": 0,
                              "Issues": 1, "Status": "ERROR", "Note": "Required column is absent; dependent checks are skipped."})
    add("Nonempty dataset", "all", bad=pd.Series(n == 0, index=df.index))
    if n == 0:
        summaries[-1].update(Issues=1, Status="ERROR", Note="No records to validate.")
    for c in REQUIRED:
        if c in df.columns:
            add("Required values", c, missing[c], expected="Nonempty value")
    for c in FLAGS:
        if c in df.columns:
            add("Required binary flag", c, missing[c], expected="0 or 1")
    for c, values in numeric.items():
        provided = ~missing[c]
        invalid = provided & (values.isna() | ~np.isfinite(values))
        add("Numeric and finite", c, invalid, eligible=provided, expected="Finite number")
    for c in ELEMENTS + PROCESS:
        if c not in df.columns:
            continue
        add("Zero-encoded field completeness", c, missing[c], severity="WARNING" if mode == "raw" else "ERROR",
            expected="Explicit numeric value (0 where the documented rule applies)",
            note="Raw blanks are reported, not changed. A blank processing value does not by itself prove the step was absent.")
    if "Sum" in df.columns:
        add("Sum completeness", "Sum", missing["Sum"], expected="Numeric sum")
    for c in ELEMENTS:
        if c in numeric:
            valid = numeric[c].notna() & np.isfinite(numeric[c])
            add("Element range", c, (numeric[c] < 0) | (numeric[c] > 100), valid,
                expected="0 <= concentration <= 100 at.%")
    if all(c in df.columns for c in ELEMENTS):
        elements_num = pd.DataFrame({c: numeric[c] for c in ELEMENTS})
        provided_invalid = pd.DataFrame({c: ~missing[c] & (numeric[c].isna() | ~np.isfinite(numeric[c])) for c in ELEMENTS}).any(axis=1)
        element_blank = pd.DataFrame({c: missing[c] for c in ELEMENTS}).any(axis=1)
        eligible = ~provided_invalid
        if mode == "final":
            eligible &= ~element_blank
        # The raw-only interpretation of blank element cells is explicit and leaves df unchanged.
        calculated_sum = elements_num.fillna(0).sum(axis=1)
        note = "Raw mode only: missing element concentrations are interpreted as zero for this calculation." if mode == "raw" else "All element concentrations must be explicitly present."
        add("Concentration sum", "elements", (calculated_sum - 100).abs() > SUM_TOLERANCE, eligible,
            note=note, observed=calculated_sum, expected=f"100 ± {SUM_TOLERANCE} at.%")
        if "Sum" in numeric:
            valid_sum = numeric["Sum"].notna() & np.isfinite(numeric["Sum"])
            add("Stored Sum consistency", "Sum", (numeric["Sum"] - calculated_sum).abs() > SUM_FIELD_TOLERANCE,
                eligible & valid_sum, observed=numeric["Sum"], expected=calculated_sum,
                note=f"Absolute tolerance: {SUM_FIELD_TOLERANCE} at.%. {note}")
    else:
        calculated_sum = None
    for c in FLAGS:
        if c in numeric:
            valid = numeric[c].notna() & np.isfinite(numeric[c])
            add("Binary values", c, ~numeric[c].isin([0, 1]), valid, expected="0 or 1")
    if "Phase composition" in df.columns:
        phases = df["Phase composition"].map(normalise_phase)
        add("Phase label structure", "Phase composition", phases.isna(), ~missing["Phase composition"],
            expected="Nonempty phase labels", note="Split on '+', trim whitespace, normalise Unicode. Rare phase labels are allowed.")
        valid_phase = phases.notna()
        for c, expected in {
            "L12 presence": phases.map(lambda s: int("L12" in s) if s is not None else np.nan),
            "Target state": phases.map(lambda s: int(s == frozenset({"FCC", "L12"})) if s is not None else np.nan)
        }.items():
            if c in numeric:
                eligible = valid_phase & numeric[c].isin([0, 1])
                add("Phase flag consistency", c, numeric[c] != expected, eligible,
                    observed=numeric[c], expected=expected, note="Phase order is ignored; no substring matching or phase synonyms inferred.")
    else:
        phases = None
    if "L12 fraction, %" in numeric:
        fraction = numeric["L12 fraction, %"]
        valid = fraction.notna() & np.isfinite(fraction)
        add("L12 fraction range", "L12 fraction, %", (fraction < 0) | (fraction > 100), valid, expected="0 <= fraction <= 100%")
        if "L12 presence" in numeric:
            add("Positive fraction vs flag", "L12 fraction, %", (fraction > 0) & (numeric["L12 presence"] == 0),
                valid & numeric["L12 presence"].isin([0, 1]), expected="Positive fraction requires L12 presence = 1")
            add("Zero fraction with presence", "L12 fraction, %", (fraction == 0) & (numeric["L12 presence"] == 1),
                valid & numeric["L12 presence"].isin([0, 1]), severity="WARNING",
                note="Review source: zero could be rounding or a coding inconsistency; do not correct automatically.")
    for c in [x for x in PROCESS if x.endswith(", min")] + ["Cold deformation, %"]:
        if c in numeric:
            valid = numeric[c].notna() & np.isfinite(numeric[c])
            bad = numeric[c] < 0
            if c == "Cold deformation, %":
                bad |= numeric[c] > 100
            add("Processing range", c, bad, valid, expected="Nonnegative; cold deformation <= 100%")
    for step in STEPS:
        t, duration = f"{step}, °C", f"{step}, min"
        if t in numeric and duration in numeric:
            valid = numeric[t].notna() & numeric[duration].notna() & np.isfinite(numeric[t]) & np.isfinite(numeric[duration])
            bad = ((numeric[t] == 0) & (numeric[duration] > 0)) | ((numeric[t] > 0) & (numeric[duration] == 0))
            add("Step temperature-duration pair", step, bad, valid, severity="WARNING",
                observed=pd.Series(list(zip(df[t], df[duration])), index=df.index),
                note="Review the source and zero encoding; do not infer or fill unknown treatment parameters.")

    if "Paper" in df.columns:
        paper_num = pd.to_numeric(df["Paper"].mask(missing["Paper"]), errors="coerce")
        is_literature_id = (paper_num.notna() & np.isfinite(paper_num)
                            & (paper_num > 0) & (paper_num % 1 == 0))
        valid = ~missing["Paper"]
        add("Paper identifier", "Paper", ~(is_literature_id | sample_mask), valid,
            expected="Positive integer (literature) or S<number> (this work)")
        if "DOI" in df.columns:
            doi_norm = df["DOI"].map(normalise_doi)
            add("Sample ID vs DOI", "Paper", sample_mask != (doi_norm == SELF_DOI),
                valid & ~missing["DOI"],
                expected="Paper = S<number> if and only if DOI = 'This work'",
                note="Experimental samples must be labelled S<number> and carry DOI 'This work'; "
                     "literature records must have a real DOI and an integer Paper.")

    duplicate_groups = pd.DataFrame()
    dup_columns = ELEMENTS + PROCESS + COOLING + ["Preparation", "DOI", "Phase composition"]
    if all(c in df.columns for c in dup_columns):
        key = pd.DataFrame(index=df.index)
        usable = pd.Series(True, index=df.index)
        for c in ELEMENTS + PROCESS:
            value = numeric[c]
            usable &= ~(~missing[c] & (value.isna() | ~np.isfinite(value)))
            value = value.round(DUPLICATE_DECIMALS)
            if c in ELEMENTS and mode == "raw":
                value = value.mask(missing[c], 0)
            key[c] = value
        for c in COOLING + ["Preparation"]:
            key[c] = df[c].map(lambda v: None if is_missing(v) else str(v).strip().casefold())
        key["DOI"] = df["DOI"].map(normalise_doi)
        usable &= ~missing["DOI"] & ~missing["Preparation"]
        key["Phase composition"] = phases.map(lambda s: tuple(sorted(s)) if s is not None else None)
        usable &= phases.notna()
        # Unknown processing/cooling values may hide real differences; all matches require review.
        condition_cols = [c for c in key.columns if c != "Phase composition"]
        duplicated = pd.Series(False, index=df.index)
        exact = pd.Series(False, index=df.index)
        if usable.any():
            candidates = key.loc[usable]
            duplicated.loc[usable] = candidates.duplicated(subset=condition_cols, keep=False)
            exact.loc[usable] = candidates.duplicated(keep=False)
        add("Potential duplicate records", "composition + processing + source", duplicated, usable,
            severity="WARNING", note=f"Same source and condition; includes matching and conflicting phases. Numeric key rounded to {DUPLICATE_DECIMALS} decimals. Missing processing values remain unknown. Manual review required.")
        add("Matching record content", "composition + processing + phases + source", exact, usable,
            severity="WARNING", note="Candidate duplicates, not automatically proven errors; flags, fraction and other measurements may differ.")
        if duplicated.any():
            key_string = key[condition_cols].astype(str).agg("|".join, axis=1)
            group_id = pd.factorize(key_string[duplicated], sort=False)[0] + 1
            duplicate_groups = df.loc[duplicated].copy()
            duplicate_groups.insert(0, "Excel_row", excel_first_data_row + duplicate_groups.index)
            duplicate_groups.insert(0, "Duplicate_group", group_id)
    else:
        summaries.append({"Check": "Potential duplicate records", "Field": "multiple", "Rows_evaluated": 0,
                          "Issues": 0, "Status": "NOT_EVALUATED", "Note": "Some columns required for the comparison are absent."})

    summary = pd.DataFrame(summaries)
    issue_columns = ["Check", "Severity", "Field", "Record_position", "Excel_row", "Observed", "Expected", "Note"] + REQUIRED
    issues = pd.DataFrame(details, columns=issue_columns)
    completeness = pd.DataFrame([{"Field": c, "Rows": n, "Missing": int(missing[c].sum()),
                                  "Present": int((~missing[c]).sum())} for c in df.columns])
    metadata = {"mode": mode, "records": n, "columns": len(df.columns), "header_aliases": renamed,
                "error_checks": int((summary.Status == "ERROR").sum()),
                "warning_checks": int((summary.Status == "WARNING").sum()),
                "not_evaluated_checks": int((summary.Status == "NOT_EVALUATED").sum()),
                "note": "Counts of issues are not counts of unique erroneous records. Source verification is not tested by this notebook."}

    # Literature papers and DOIs are counted over literature records only
    literature_mask = ~sample_mask
    metadata["literature_records"] = int(literature_mask.sum())
    metadata["experimental_records"] = int(sample_mask.sum())
    if "Paper" in df.columns:
        metadata["unique_literature_papers"] = int(paper_text[literature_mask & ~missing["Paper"]].nunique())
        metadata["unique_experimental_samples"] = int(paper_text[sample_mask].nunique())
    if "DOI" in df.columns:
        metadata["unique_literature_dois"] = int(
            df.loc[literature_mask & ~missing["DOI"], "DOI"].map(normalise_doi).nunique())

    if calculated_sum is not None:
        finite_sums = calculated_sum[~provided_invalid & (~element_blank if mode == "final" else True)]
        finite_sums = finite_sums[np.isfinite(finite_sums)]
        metadata["max_sum_deviation_descriptive"] = float((finite_sums - 100).abs().max()) if len(finite_sums) else None
    return summary, issues, completeness, duplicate_groups, metadata


In [3]:
def main():
    if not SOURCE.is_file():
        raise FileNotFoundError(f"Set SOURCE to an existing dataset file: {SOURCE}")
    if SOURCE.suffix.lower() in {".xlsx", ".xlsm"}:
        raw = pd.read_excel(SOURCE, sheet_name=SHEET_NAME, header=HEADER_ROW,
                            dtype=object, keep_default_na=False)
        first_data_row = HEADER_ROW + 2
    elif SOURCE.suffix.lower() == ".parquet":
        raw = pd.read_parquet(SOURCE)
        first_data_row = 2
    else:
        raise ValueError("Supported formats: .xlsx, .xlsm, .parquet")
    summary, issues, completeness, duplicates, metadata = validate_dataset(raw, MODE, first_data_row)
    REPORT_DIR.mkdir(parents=True, exist_ok=True)
    metadata.update({"source_name": SOURCE.name,
                     "source_sha256": hashlib.sha256(SOURCE.read_bytes()).hexdigest(),
                     "utc_time": datetime.now(timezone.utc).isoformat(),
                     "python": platform.python_version(), "pandas": pd.__version__, "numpy": np.__version__,
                     "sheet": SHEET_NAME, "header_row_zero_based": HEADER_ROW,
                     "sum_tolerance_at_percent": SUM_TOLERANCE,
                     "sum_field_tolerance": SUM_FIELD_TOLERANCE,
                     "duplicate_decimals": DUPLICATE_DECIMALS,
                     "row_locator": "Excel_row is applicable to Excel input; use Record_position for Parquet."})
    tables = {"Checks": summary, "Issues": issues, "Completeness": completeness, "Duplicate_candidates": duplicates}
    with pd.ExcelWriter(REPORT_DIR / "validation_report.xlsx", engine="openpyxl") as writer:
        for name, table in tables.items():
            table.to_excel(writer, sheet_name=name, index=False)
            ws = writer.sheets[name]
            ws.freeze_panes = "A2"
            if len(table.columns):
                ws.auto_filter.ref = ws.dimensions
            for cells in ws.columns:
                width = max(len(str(cell.value or "")) for cell in list(cells)[:200])
                ws.column_dimensions[cells[0].column_letter].width = min(max(width + 2, 12), 60)
    summary.to_csv(REPORT_DIR / "validation_summary.csv", index=False, encoding="utf-8-sig")
    issues.to_csv(REPORT_DIR / "validation_issues.csv", index=False, encoding="utf-8-sig")
    (REPORT_DIR / "validation_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
    display(pd.DataFrame([metadata])[['records', 'literature_records', 'experimental_records',
                                      'unique_literature_papers', 'unique_experimental_samples',
                                      'unique_literature_dois', 'columns', 'error_checks',
                                      'warning_checks', 'not_evaluated_checks']])
    display(summary)
    if len(issues):
        display(issues.head(30))
    return summary, issues, completeness, duplicates, metadata

if __name__ == "__main__":
    results = main()

,records,literature_records,experimental_records,unique_literature_papers,unique_experimental_samples,unique_literature_dois,columns,error_checks,warning_checks,not_evaluated_checks
0,386,374,12,74,12,74,38,0,0,0


,Check,Field,Rows_evaluated,Issues,Status,Note
0,Nonempty dataset,all,386,0,PASS,
1,Required values,Paper,386,0,PASS,
2,Required values,"Composition, at.%",386,0,PASS,
3,Required values,Preparation,386,0,PASS,
4,Required values,Phase composition,386,0,PASS,
...,...,...,...,...,...,...
98,Step temperature-duration pair,Aging 2,386,0,PASS,Review the source and zero encoding; do not in...
99,Paper identifier,Paper,386,0,PASS,
100,Sample ID vs DOI,Paper,386,0,PASS,Experimental samples must be labelled S<number...
101,Potential duplicate records,composition + processing + source,386,0,PASS,Same source and condition; includes matching a...
